## EDA & Data Cleansing

### References: https://www.kaggle.com/code/jagangupta/stop-the-s-toxic-comments-eda

### Used the resource above as a baseline + Modified it so that it is compatiable with latest versions

#### In case of NLTK errors: <python -m nltk.downloader wordnet>

## Install required packages in the following cell before proceed

In [ ]:
import pandas as pd 
import numpy as np

import gc
import time
import warnings

from imageio import imread
from scipy import sparse
import scipy.stats as ss

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec 
import seaborn as sns
from wordcloud import WordCloud ,STOPWORDS
from PIL import Image
import matplotlib_venn as venn

import string
import re
import nltk
from nltk.corpus import stopwords
import spacy
from nltk import pos_tag
from nltk.stem.wordnet import WordNetLemmatizer 
from nltk.tokenize import word_tokenize
from nltk.tokenize import TweetTokenizer   

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, HashingVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_is_fitted
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

start_time=time.time()
color = sns.color_palette()
sns.set_style("dark")
eng_stopwords = set(stopwords.words("english"))
warnings.filterwarnings("ignore")

lem = WordNetLemmatizer()
tokenizer=TweetTokenizer()

%matplotlib inline

## Modify the path with your own setting

In [ ]:
train=pd.read_csv("./jigsaw-toxic-comment-classification-challenge/train.csv/train.csv")
test=pd.read_csv("./jigsaw-toxic-comment-classification-challenge/test.csv/test.csv")

In [ ]:
train.head(10)

In [ ]:
n_train = train.shape[0]
n_test = test.shape[0]
n_total = n_train + n_test
print("       | Train | Test")
print("count  :",n_train,":",n_test)
print("ratio  :", round(n_train * 100 / n_total),"   :",round(n_test * 100 / n_total))

In [ ]:
x = train.iloc[:, 2:].sum()
train['non-toxic'] = (train.iloc[:,2:].sum(axis=1)==0)
print("Total : ", len(train))
print("Total non-toxic comments : ", train['non-toxic'].sum())
print("Total labels : ", x.sum())

### Seems like we have a pretty biased dataset. We should keep it in mind when training.

In [ ]:
print('Check Null Values: ')
print(train.isnull().sum())

In [ ]:
x = train.iloc[:,2:].sum()
plt.figure(figsize=(12,6))
ax = sns.barplot(x = x.index, y = x.values)
plt.ylabel('Count', fontsize=10)
plt.xlabel('Class ', fontsize=10)
rects = ax.patches
labels = x.values
for rect, label in zip(rects, labels):
    height = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2, height + 5, label, ha='center', va='bottom')
plt.show()

In [ ]:
x = train.iloc[:,2:].sum(axis=1).value_counts()

plt.figure(figsize=(12,6))
ax = sns.barplot(x = x.index, y = x.values)
plt.ylabel('Count', fontsize=10)
plt.xlabel('# of labels', fontsize=10)

rects = ax.patches
labels = x.values
for rect, label in zip(rects, labels):
    height = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2, height + 5, label, ha='center', va='bottom')

plt.show()

### Some datapoints have multiple tags.

In [ ]:
temp_df = train.iloc[:,2:-1]
mats = []
for col in temp_df.columns[1:]:
    mat = pd.crosstab(temp_df["toxic"], temp_df[col])
    mats.append(mat)
mats = pd.concat(mats, axis=1, keys=temp_df.columns[1:])
mats

### Seems like toxic=0 even if a comment is obscene/threat/...
### Maybe we should apply toxic=1 if a comment is obscene/threat/...

In [ ]:
train.loc[(train['severe_toxic']==1) | (train['obscene']==1) | (train['threat']==1) | (train['insult']==1) | (train['identity_hate']==1), 'toxic']=1

In [ ]:
temp_df = train.iloc[:,2:-1]
mats = []
for col in temp_df.columns[1:]:
    mat = pd.crosstab(temp_df["toxic"], temp_df[col])
    mats.append(mat)
mats = pd.concat(mats, axis=1, keys=temp_df.columns[1:])
mats

In [ ]:
print("toxic:")
print(train[train.severe_toxic==1].iloc[3,1])

In [ ]:
print("severe_toxic:")
print(train[train.severe_toxic==1].iloc[4,1])

In [ ]:
print("Threat:")
print(train[train.threat==1].iloc[1,1])

In [ ]:
print("Obscene:")
print(train[train.obscene==1].iloc[1,1])

In [ ]:
print("identity_hate:")
print(train[train.identity_hate==1].iloc[4,1])

In [ ]:
train.describe()

In [ ]:
cols = ['obscene', 'insult', 'toxic', 'severe_toxic', 'identity_hate', 'threat']
print(train[cols].sum())

In [ ]:
train['length'] = train['comment_text'].apply(lambda x: len(str(x)))
train['length'].hist()
plt.show()

### Most of the text length are within 500 characters, with some up to 5,000 characters long.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "cannot ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub(r"\'scuse", " excuse ", text)
    text = re.sub('\W', ' ', text)
    text = re.sub('\s+', ' ', text)
    text = text.strip(' ')
    return text

In [ ]:
train['comment_text'] = train['comment_text'].map(lambda com : clean_text(com))
test['comment_text'] = test['comment_text'].map(lambda com : clean_text(com))

In [ ]:
train['comment_text'][0]

In [ ]:
train['comment_text'][1]

In [ ]:
train['comment_text'][2]

In [ ]:
train['comment_text'][10]

In [ ]:
train['comment_text'][10000]

In [ ]:
train['comment_text'][100000]

In [ ]:
train.head(20)

In [ ]:
test.head(20)

## Until here what we did: EDA + Cleansing text